# **Fine-Tuning a Legal Assistant LLM**

# 1 - Installing required packages

In [1]:
!pip install -q -U watermark

In [2]:
!pip install -q -r /content/drive/MyDrive/AI_Learning/1_Generative_AI/Projects/Cap09/requirements.txt

In [3]:
# Imports
import numpy as np
import nltk
import shutil
import evaluate
import transformers
import datasets
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers import DataCollatorForSeq2Seq, T5Tokenizer
from transformers import T5ForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer

In [4]:
# Versions used in jupyter notebook
%reload_ext watermark
%watermark -a "João Machado"

Author: João Machado



In [5]:
# Progamatically deleting folders to avoid issues
try:
  shutil.rmtree('/content/drive/MyDrive/AI_Learning/1_Generative_AI/Projects/Cap09/logs_train')
  shutil.rmtree('/content/drive/MyDrive/AI_Learning/1_Generative_AI/Projects/Cap09/results_train')
  shutil.rmtree('/content/drive/MyDrive/AI_Learning/1_Generative_AI/Projects/Cap09/saved_model')
except:
    print("No folders found")

No folders found


# 2 - Loading dataset

---



In [6]:
# Defining archive name
archive_name = "/content/drive/MyDrive/AI_Learning/1_Generative_AI/Projects/Cap09/dataset.csv"

In [7]:
# Loading the data
law_dataset = load_dataset("csv", data_files=archive_name, delimiter = ',')
law_dataset

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 3742
    })
})

In [8]:
# Dividing train and test with proportion of 80/20
law_dataset = law_dataset["train"].train_test_split(test_size=0.2)
law_dataset

DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 2993
    })
    test: Dataset({
        features: ['question', 'answer'],
        num_rows: 749
    })
})

# 3 - Loading Tokenizer and LLM

In [9]:
# Loading tokenizer
tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-base", legacy = False)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [10]:
# See tokenizer
tokenizer

T5Tokenizer(name_or_path='google/flan-t5-base', vocab_size=32000, model_max_length=512, is_fast=False, padding_side='right', truncation_side='right', special_tokens={'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<pad>', 'additional_special_tokens': ['<extra_id_0>', '<extra_id_1>', '<extra_id_2>', '<extra_id_3>', '<extra_id_4>', '<extra_id_5>', '<extra_id_6>', '<extra_id_7>', '<extra_id_8>', '<extra_id_9>', '<extra_id_10>', '<extra_id_11>', '<extra_id_12>', '<extra_id_13>', '<extra_id_14>', '<extra_id_15>', '<extra_id_16>', '<extra_id_17>', '<extra_id_18>', '<extra_id_19>', '<extra_id_20>', '<extra_id_21>', '<extra_id_22>', '<extra_id_23>', '<extra_id_24>', '<extra_id_25>', '<extra_id_26>', '<extra_id_27>', '<extra_id_28>', '<extra_id_29>', '<extra_id_30>', '<extra_id_31>', '<extra_id_32>', '<extra_id_33>', '<extra_id_34>', '<extra_id_35>', '<extra_id_36>', '<extra_id_37>', '<extra_id_38>', '<extra_id_39>', '<extra_id_40>', '<extra_id_41>', '<extra_id_42>', '<extra_id_43>', '

In [11]:
# Loading LLM
model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-base")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [12]:
# See model arquitecture
model

T5ForConditionalGeneration(
  (shared): Embedding(32128, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=768, out_features=2048, bias=False)
              (wi_1): Linear(in_features=768, out_features=2048, bias=False)
              (wo):

In [13]:
# Data collator for model and tokenizer concatenation (This is like a pipeline)
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

In [14]:
# See data collator
data_collator

DataCollatorForSeq2Seq(tokenizer=T5Tokenizer(name_or_path='google/flan-t5-base', vocab_size=32000, model_max_length=512, is_fast=False, padding_side='right', truncation_side='right', special_tokens={'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<pad>', 'additional_special_tokens': ['<extra_id_0>', '<extra_id_1>', '<extra_id_2>', '<extra_id_3>', '<extra_id_4>', '<extra_id_5>', '<extra_id_6>', '<extra_id_7>', '<extra_id_8>', '<extra_id_9>', '<extra_id_10>', '<extra_id_11>', '<extra_id_12>', '<extra_id_13>', '<extra_id_14>', '<extra_id_15>', '<extra_id_16>', '<extra_id_17>', '<extra_id_18>', '<extra_id_19>', '<extra_id_20>', '<extra_id_21>', '<extra_id_22>', '<extra_id_23>', '<extra_id_24>', '<extra_id_25>', '<extra_id_26>', '<extra_id_27>', '<extra_id_28>', '<extra_id_29>', '<extra_id_30>', '<extra_id_31>', '<extra_id_32>', '<extra_id_33>', '<extra_id_34>', '<extra_id_35>', '<extra_id_36>', '<extra_id_37>', '<extra_id_38>', '<extra_id_39>', '<extra_id_40>', '<extra_id_41>', '<

# 4 - Pre-processing

In [15]:
# Defining prefix
prefix = "answer the question: "

In [16]:
# Function for pre-processing
def fn_preprocessing(data):

  # Add prefix to each question
  inputs = [prefix + doc for doc in data["question"]]

  # Tokens the questions
  model_inputs = tokenizer(inputs, max_length=128, truncation=True)

  # Tokens the answers
  labels = tokenizer(text_target=data["answer"], max_length=512, truncation=True)

  # Add them together
  model_inputs["labels"] = labels["input_ids"]

  return model_inputs

In [17]:
# Applies the function to dataset
law_dataset_tokenized = law_dataset.map(fn_preprocessing, batched=True)

Map:   0%|          | 0/2993 [00:00<?, ? examples/s]

Map:   0%|          | 0/749 [00:00<?, ? examples/s]

In [18]:
# See data
law_dataset_tokenized

DatasetDict({
    train: Dataset({
        features: ['question', 'answer', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 2993
    })
    test: Dataset({
        features: ['question', 'answer', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 749
    })
})

In [19]:
## See human input vs tokenize input
# Human question
law_dataset_tokenized['train']['question'][0]

"Q: Is there a legal form that can be used to guarantee my brother share our mother's settlement?. My mother was a Psychiatric Technician at Patton State Hospital. She was injured at work in 2004. She was deemed disabled and medically retired due to her injuries sustained at Patton. She initiated a lawsuit in 2005 against workers comp/state fund. She passed away in September of 2020. In October of 2023, my older brother and I attended a court hearing over the phone, and my mother won the lawsuit 3 years after her death. My brother is asking that I sign for him to be in charge of the money so we avoid probate, but I am concerned with this. Is there any way I can get a notarized agreement that will have any legal standing in the event that he decide not to give me my half of our moms money? "

In [20]:
## See human input vs tokenize input
# Human answer
law_dataset_tokenized['train']['answer'][0]

"A:Assuming your mother lived in California, the response to your question can be ascertained once you answer two questions: (1) Did your mother have a Trust or Will? (2) What is the collective dollar value of your mother's assets as of the date of her death? If her assets were valued at $154,500, you should see an attorney about the legal requirements for probate. If her assets are valued at less than that amount, there is an affidavit that you can sign, but it must contain specific language required by law, which is too long to put in this answer. I hope that helps!"

In [21]:
## See human input vs tokenize input
# tokenize question
law_dataset_tokenized['train']['input_ids'][0]

[1525,
 8,
 822,
 10,
 1593,
 10,
 27,
 7,
 132,
 3,
 9,
 1281,
 607,
 24,
 54,
 36,
 261,
 12,
 3614,
 82,
 4284,
 698,
 69,
 2039,
 31,
 7,
 7025,
 58,
 5,
 499,
 2039,
 47,
 3,
 9,
 3,
 21513,
 23,
 9,
 3929,
 26993,
 44,
 5192,
 17,
 106,
 1015,
 4457,
 5,
 451,
 47,
 7532,
 44,
 161,
 16,
 4406,
 5,
 451,
 47,
 3,
 10863,
 10860,
 11,
 1035,
 120,
 10611,
 788,
 12,
 160,
 5157,
 14399,
 44,
 5192,
 17,
 106,
 5,
 451,
 16781,
 3,
 9,
 9953,
 16,
 3105,
 581,
 2765,
 2890,
 87,
 5540,
 3069,
 5,
 451,
 2804,
 550,
 16,
 1600,
 13,
 6503,
 5,
 86,
 1797,
 13,
 460,
 2773,
 6,
 82,
 2749,
 4284,
 11,
 27,
 5526,
 3,
 9,
 1614,
 3507,
 147,
 8,
 951,
 6,
 11,
 82,
 2039,
 751,
 8,
 9953,
 220,
 203,
 227,
 160,
 1687,
 1]

In [22]:
## See human input vs tokenize input
# tokenize answer
law_dataset_tokenized['train']['labels'][0]

[71,
 10,
 188,
 7,
 4078,
 53,
 39,
 2039,
 4114,
 16,
 1826,
 6,
 8,
 1773,
 12,
 39,
 822,
 54,
 36,
 38,
 2110,
 10733,
 728,
 25,
 1525,
 192,
 746,
 10,
 5637,
 3963,
 39,
 2039,
 43,
 3,
 9,
 5313,
 42,
 2003,
 58,
 6499,
 363,
 19,
 8,
 6018,
 6816,
 701,
 13,
 39,
 2039,
 31,
 7,
 4089,
 38,
 13,
 8,
 833,
 13,
 160,
 1687,
 58,
 156,
 160,
 4089,
 130,
 12695,
 44,
 15287,
 8525,
 2560,
 6,
 25,
 225,
 217,
 46,
 4917,
 81,
 8,
 1281,
 1502,
 21,
 12361,
 342,
 5,
 156,
 160,
 4089,
 33,
 12695,
 44,
 705,
 145,
 24,
 866,
 6,
 132,
 19,
 46,
 3,
 4127,
 23,
 26,
 9,
 5566,
 24,
 25,
 54,
 1320,
 6,
 68,
 34,
 398,
 3480,
 806,
 1612,
 831,
 57,
 973,
 6,
 84,
 19,
 396,
 307,
 12,
 474,
 16,
 48,
 1525,
 5,
 27,
 897,
 24,
 1691,
 55,
 1]

# 5 - Defining Evaluation Metric

In [23]:
# Using 'punkt' for tokenization --> for text into sentence list
nltk.download('punkt', quiet = True)

True

In [24]:
# Download secundary package 'punkt_tab'
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [25]:
# Defining eval metric
metric = evaluate.load("rouge")

In [26]:
# Defining calculation to apply the metric
def law_metrics_calculation(eval_pred):

    # Passing the labels and preds to separate vars
    predictions, labels = eval_pred

    # Substitute -100 values with Tokenizer ID (if its -100 then its not important, it was just used for padding when we did the tokenization)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    # Decoding predictions to text
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)

    # Decoding labels to text
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Applying special transformation since ROUGE eval metric requires it - for predictions
    decoded_preds_transformed = ["/n".join(nltk.sent_tokenize(pred.strip())) for pred in decoded_preds]

    # Applying special transformation since ROUGE eval metric requires it - for labels
    decoded_labels_transformed = ["/n".join(nltk.sent_tokenize(label.strip())) for label in decoded_labels]

    # Applying the eval metric (ROUGE)
    result = metric.compute(
        predictions = decoded_preds_transformed,
        references  = decoded_labels_transformed,
        use_stemmer = True,
    )

    # Returning result
    return result

# 6 - Fine-tuning the model

In [27]:
# ### V1
# # Defining hiper-parameters for Trainer
# law_training_args = Seq2SeqTrainingArguments(
#     output_dir                  = "/content/drive/MyDrive/AI_Learning/1_Generative_AI/Projects/Cap09/results_train",
#     eval_strategy               = "epoch",
#     learning_rate               = 3e-4,
#     per_device_train_batch_size = 4,
#     per_device_eval_batch_size  = 2,
#     weight_decay                = 0.01,
#     save_total_limit            = 3,
#     num_train_epochs            = 3,
#     predict_with_generate       = True,
#     push_to_hub                 = False
# )

In [28]:
### V2
# Defining hiper-parameters for Trainer
law_training_args = Seq2SeqTrainingArguments(
    output_dir                  = "/content/drive/MyDrive/AI_Learning/1_Generative_AI/Projects/Cap09/results_train",
    eval_strategy               = "epoch",
    learning_rate               = 3e-4,
    per_device_train_batch_size = 8,
    per_device_eval_batch_size  = 4,
    weight_decay                = 0.01,
    save_total_limit            = 3,
    num_train_epochs            = 6,
    predict_with_generate       = True,
    push_to_hub                 = False
)

In [29]:
# Define Trainer
law_trainer = Seq2SeqTrainer(
      model           = model,
      args            = law_training_args,
      train_dataset   = law_dataset_tokenized["train"],
      eval_dataset    =  law_dataset_tokenized["test"],
      tokenizer       = tokenizer,
      data_collator   = data_collator,
      compute_metrics = law_metrics_calculation
)

/tmp/ipykernel_17351/1948008704.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  law_trainer = Seq2SeqTrainer(


In [30]:
%%time
law_trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1,No log,2.439477,0.125975,0.030266,0.101433,0.101183
2,2.659900,2.402913,0.124665,0.029306,0.100896,0.100719
3,2.379900,2.389042,0.125653,0.030495,0.101178,0.101014
4,2.230800,2.399606,0.126667,0.030455,0.101850,0.101759
5,2.230800,2.400728,0.126251,0.030471,0.101237,0.101151
6,2.092000,2.407730,0.125480,0.030008,0.100550,0.100461


CPU times: user 22min 37s, sys: 2min 9s, total: 24min 46s
Wall time: 25min 38s


TrainOutput(global_step=2250, training_loss=2.3104422743055557, metrics={'train_runtime': 1538.0871, 'train_samples_per_second': 11.676, 'train_steps_per_second': 1.463, 'total_flos': 3060085651504128.0, 'train_loss': 2.3104422743055557, 'epoch': 6.0})

In [31]:
# Save the model
law_trainer.save_model("/content/drive/MyDrive/AI_Learning/1_Generative_AI/Projects/Cap09/saved_model")

# 7 - Deploying and Using the Model

In [32]:
# Loading saved tokenizer
tokenizer_loaded = AutoTokenizer.from_pretrained("/content/drive/MyDrive/AI_Learning/1_Generative_AI/Projects/Cap09/saved_model")

You set `add_prefix_space`. The tokenizer needs to be converted from the slow tokenizers


In [33]:
# Loading saved model
model_loaded = AutoModelForSeq2SeqLM.from_pretrained("/content/drive/MyDrive/AI_Learning/1_Generative_AI/Projects/Cap09/saved_model")

In [34]:
# Visualizing model
model_loaded

T5ForConditionalGeneration(
  (shared): Embedding(32128, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=768, out_features=2048, bias=False)
              (wi_1): Linear(in_features=768, out_features=2048, bias=False)
              (wo):

In [35]:
# Input for model
question_input = "In US when there's a divorce of 2 people, is always mandatory to pay child support when the couple have children? If not, tell me 3 big situations of why not"

In [36]:
# Tokenizing input
question_input_tokenized = tokenizer_loaded(question_input, return_tensors="pt").input_ids

In [37]:
# Prediction
answer_tokenized = model_loaded.generate(question_input_tokenized, max_length=512, temperature=0.4, do_sample=True)

In [38]:
# Seeing tokenized prediction result by model
answer_tokenized

tensor([[    0,  9364,   380,    19,    59, 13488,    16,     8,   907,  1323,
             6,    68,    34,    19,     3,     9,  1017,  1032,    16,   186,
         10185,     7,     5,    86,   128,  1488,     6,    34,    19,    59,
         13488,     6,    68,    34,    54,    36,     3,     9,  1516,  2945,
            16,     8, 16701,    13,     8,   502,     5,    86,   175,  4147,
             6,     8,  1614,   164,   455,    24,     8,  1362,   726,   861,
           380,     3,    99,     8,   502,    33,    59,   502,     5,   100,
            19,     3,     9,  1017,  1032,    16,   384,   973,     6,   902,
            16,  1488,     3,  6475, 16701, 18126,     5,    86,     8,   907,
          1323,     6,   861,   380,    19,     3,     9,  1281,  5971,    21,
           321,  1362,     6,  6147,    13,    70,  1675,  2637,     5,   611,
             6,    16,   128,  4147,     6,    34,   164,    59,    36, 13488,
             5,    86,  1488,   213,   132,    19,  

In [39]:
# Decode prediction (to understand it)
answer_detokenized = tokenizer_loaded.decode(answer_tokenized[0], skip_special_tokens=True)

In [40]:
# Result
print("Question: ", question_input)
print("Answer: ", answer_detokenized)

Question:  In US when there's a divorce of 2 people, is always mandatory to pay child support when the couple have children? If not, tell me 3 big situations of why not
Answer:  Child support is not mandatory in the United States, but it is a common practice in many jurisdictions. In some cases, it is not mandatory, but it can be a significant factor in the custody of the children. In these situations, the court may order that the parents pay child support if the children are not children. This is a common practice in family law, especially in cases involving custody disputes. In the United States, child support is a legal requirement for both parents, regardless of their relationship status. However, in some situations, it may not be mandatory. In cases where there is a custody dispute, the court may order that the parents pay child support. This is a way to ensure that the children are not deprived of their parents' rights. In cases where there is a custody dispute, the court may ord